# 7장 선형 회귀 — 최소제곱법, 다중 선형 회귀, 회귀 지표, 강건한 손실
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 기울기를 바꿔 가며 MSE 재기, 최소제곱 공식으로 직선 찾기
2. 펭귄 몸무게를 특징 여러 개로 예측하고 계수 읽기
3. MAE, MSE, RMSE, 결정 계수($R^2$)로 회귀 모델 채점하기
4. 잘못 적힌 기록이 섞인 택시 데이터에서 MSE, MAE, 후버 손실 비교하기

위에서부터 차례로 실행하세요.

## 1. 가장 잘 맞는 직선 찾기
기록 세 건(예시)에서 절편을 $b = 4$로 두고 기울기 $w$만 바꾸어 MSE를 잽니다.

In [ ]:
import numpy as np

x = np.array([2.0, 4.0, 6.0])     # 거리(km)
y = np.array([7.0, 8.0, 12.0])    # 요금(천 원)
for w in [0.75, 1.25, 1.75]:      # 절편은 b = 4로 둔다
    y_hat = w * x + 4
    mse = np.mean((y - y_hat) ** 2)
    print(f"w = {w}: 예측 {y_hat}, MSE {mse:.3f}")

$w$를 촘촘하게 바꾸며 MSE를 그리면 그릇 모양이 됩니다(그래프 글자는 한글 글꼴이 없는 Colab에서도 보이도록 영어로 적었습니다).

In [ ]:
import matplotlib.pyplot as plt

ws = np.linspace(0.2, 2.3, 200)
plt.plot(ws, [np.mean((y - (w * x + 4)) ** 2) for w in ws])
plt.xlabel("w"); plt.ylabel("MSE"); plt.title("b = 4")
plt.show()

그릇의 바닥을 최소제곱 공식 $w = \sum (x_i - \bar{x})(y_i - \bar{y}) / \sum (x_i - \bar{x})^2$, $b = \bar{y} - w\bar{x}$로 구합니다.

In [ ]:
dx, dy = x - x.mean(), y - y.mean()   # 평균에서 떨어진 정도
w = (dx * dy).sum() / (dx ** 2).sum()
b = y.mean() - w * x.mean()
print("w =", w, " b =", b)

1장의 택시 기록 50건에 같은 공식을 쓰고 `LinearRegression`과 비교합니다.

In [ ]:
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(seed=7)        # 1장과 같은 기록 50건
km = rng.uniform(1, 20, size=50).round(1)
fare = (1000 * km + 4800 + rng.normal(0, 600, size=50)).round(-1)

dx, dy = km - km.mean(), fare - fare.mean()
w = (dx * dy).sum() / (dx ** 2).sum()
b = fare.mean() - w * km.mean()
print(f"공식으로:          w = {w:.2f}, b = {b:.2f}")
model = LinearRegression().fit(km.reshape(-1, 1), fare)
print(f"LinearRegression: w = {model.coef_[0]:.2f}, "
      f"b = {model.intercept_:.2f}")

이북 7.2절: 기울기는 상관계수 $r$에 두 표준편차의 비를 곱한 것과 같습니다($w = r \times s_y / s_x$). 펭귄의 날개 길이와 몸무게로 확인합니다.

In [ ]:
import seaborn as sns

pg = sns.load_dataset("penguins").dropna(subset=["flipper_length_mm", "body_mass_g"])
r = pg["flipper_length_mm"].corr(pg["body_mass_g"])     # 3장의 상관계수
sx, sy = pg["flipper_length_mm"].std(), pg["body_mass_g"].std()
print(f"r = {r:.3f}, x의 표준편차 {sx:.2f}, y의 표준편차 {sy:.2f}")
print(f"r × sy / sx = {r * sy / sx:.2f}")
w2 = LinearRegression().fit(pg[["flipper_length_mm"]], pg["body_mass_g"]).coef_[0]
print(f"LinearRegression의 기울기 = {w2:.2f}")

## 2. 특징이 여러 개일 때: 다중 선형 회귀
펭귄 342마리를 학습 데이터와 테스트 데이터로 나누고(종의 비율을 맞춤), 날개 길이, 부리 길이, 부리 깊이로 몸무게를 예측합니다.

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=num)
df = df.reset_index(drop=True)                  # 342마리
feats = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
tr, te = train_test_split(df, test_size=0.25, random_state=42,
                          stratify=df["species"])
reg = LinearRegression().fit(tr[feats], tr["body_mass_g"])
print(pd.Series(reg.coef_, index=feats).round(2))
print("intercept:", round(reg.intercept_, 2))

부리 깊이 하나만 쓰면 계수의 부호가 어떻게 되나요? 세 특징을 함께 쓸 때와 비교하세요(3장의 심슨의 역설).

In [ ]:
one = LinearRegression().fit(tr[["bill_depth_mm"]],
                             tr["body_mass_g"])
print("부리 깊이 하나만 쓸 때의 계수:", one.coef_.round(1))

특징을 표준화하고 학습하면 계수를 서로 비교할 수 있습니다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

std = make_pipeline(StandardScaler(), LinearRegression())
std.fit(tr[feats], tr["body_mass_g"])
print(pd.Series(std[-1].coef_, index=feats).round(1))

종을 원-핫 인코딩으로 넣습니다. 첫 번째 종(아델리)의 열은 빼서 기준으로 삼습니다.

In [ ]:
sp = pd.get_dummies(df["species"], drop_first=True, dtype=int)
df2 = pd.concat([df, sp], axis=1)     # Chinstrap, Gentoo 열
feats2 = feats + ["Chinstrap", "Gentoo"]
tr2, te2 = df2.loc[tr.index], df2.loc[te.index]  # 같은 나눔
reg2 = LinearRegression().fit(tr2[feats2], tr2["body_mass_g"])
print(pd.Series(reg2.coef_, index=feats2).round(1))

## 3. 회귀 모델의 성적표
2장의 택시 기록 세 건과 규칙 B(1,000원 × 거리 + 5,000원)로 네 지표를 계산합니다. 이북의 손 계산(MAE 473.3, MSE 364,400, RMSE 603.7, $R^2$ 0.988)과 같은지 확인하세요.

참고: `root_mean_squared_error`는 사이킷런 1.4부터 있습니다. 그보다 오래된 버전이면 `np.sqrt(mean_squared_error(...))`로 계산합니다.

In [ ]:
from sklearn.metrics import (mean_absolute_error,
    mean_squared_error, root_mean_squared_error, r2_score)

y3 = np.array([10140, 18900, 23260])             # 2장의 세 기록
p3 = 1000 * np.array([5.3, 12.9, 18.0]) + 5000   # 규칙 B의 예측
print(f"MAE  {mean_absolute_error(y3, p3):10,.1f}")
print(f"MSE  {mean_squared_error(y3, p3):10,.1f}")
print(f"RMSE {root_mean_squared_error(y3, p3):10,.1f}")
print(f"R²   {r2_score(y3, p3):10.3f}")

펭귄 몸무게 모델들을 테스트 데이터로 채점합니다. 첫 줄은 학습 데이터의 평균만 답하는 기준 예측입니다.

In [ ]:
def report(name, y_true, y_pred):
    print(f"MAE {mean_absolute_error(y_true, y_pred):5.1f}  "
          f"RMSE {root_mean_squared_error(y_true, y_pred):5.1f}  "
          f"R² {r2_score(y_true, y_pred):6.3f}  {name}")

yte = te["body_mass_g"]
base = np.full(len(te), tr["body_mass_g"].mean())  # 늘 평균
flip = LinearRegression().fit(tr[["flipper_length_mm"]],
                              tr["body_mass_g"])
report("평균만", yte, base)
report("날개 길이", yte, flip.predict(te[["flipper_length_mm"]]))
report("특징 3개", yte, reg.predict(te[feats]))
report("특징 3개 + 종", yte, reg2.predict(te2[feats2]))
report("특징 3개 + 종(학습 데이터)", tr2["body_mass_g"],
       reg2.predict(tr2[feats2]))

## 4. 이상한 값에 덜 흔들리는 손실
요금 다섯 건 가운데 하나가 잘못 적혔을 때 평균과 중앙값이 얼마나 움직이는지 봅니다.

In [ ]:
fares = np.array([8000, 9000, 10000, 11000, 12000])
bad = fares.copy()
bad[-1] = 1200          # 12,000원을 1,200원으로 잘못 적었다
for name, v in [("원래 기록", fares), ("잘못 적힌 기록", bad)]:
    print(f"{name}: 평균 {v.mean():,.0f}원, "
          f"중앙값 {np.median(v):,.0f}원")

2장의 상황(택시 기록 50건 + 잘못 적힌 기록 3건)에서 MSE, MAE, 후버 손실로 직선을 찾습니다.

참고: 사이킷런과 사이파이(SciPy)의 버전 조합에 따라 `HuberRegressor`가 `OptimizeWarning`을 띄울 수 있지만 결과에는 영향이 없습니다.

In [ ]:
from sklearn.linear_model import HuberRegressor, QuantileRegressor

km2 = np.r_[km, [16.0, 17.5, 19.0]]     # 잘못 적힌 기록 3건
fare2 = np.r_[fare, [2100, 2250, 2400]]
X2 = km2.reshape(-1, 1)
models = {"MSE": LinearRegression(),
          "MAE": QuantileRegressor(quantile=0.5, alpha=0),
          "후버": HuberRegressor()}
X50 = km.reshape(-1, 1)                    # 원래 기록 50건
rmse0 = root_mean_squared_error(fare, model.predict(X50))  # 7.2절의 model
print(f"(잘못 적힌 기록이 없을 때 RMSE {rmse0:,.0f})")
for name, m in models.items():
    m.fit(X2, fare2)
    rmse = root_mean_squared_error(fare, m.predict(X50))
    print(f"w = {m.coef_[0]:5.1f}, b = {m.intercept_:5,.0f}, "
          f"원래 50건 RMSE {rmse:5,.0f}  {name}")
h = models["후버"]
print(f"후버의 경계: {h.epsilon} × {h.scale_:.0f} = "
      f"{h.epsilon * h.scale_:.0f}원")

세 직선을 그려 봅니다.

In [ ]:
xs = np.array([0, 21]).reshape(-1, 1)
plt.scatter(km, fare, s=15, label="records 50")
plt.scatter([16.0, 17.5, 19.0], [2100, 2250, 2400], marker="x", s=60, label="wrong 3")
for name, m in models.items():
    plt.plot(xs, m.predict(xs), label={"MSE": "MSE", "MAE": "MAE", "후버": "Huber"}[name])
plt.xlabel("km"); plt.ylabel("fare"); plt.legend(); plt.show()

---
## 실습 문제

**문제 1.** 성별(`sex`)도 특징으로 넣어 보세요. 성별이 비어 있는 펭귄을 빼면 333마리가 남습니다. 종과 성별을 원-핫 인코딩(`drop_first=True`)하고, 333마리를 2절과 같은 방법(`test_size=0.25`, `random_state=42`, 종으로 `stratify`)으로 나누어, ① 세 특징 + 종, ② 세 특징 + 종 + 성별의 두 모델을 테스트 데이터의 MAE, RMSE, $R^2$로 비교하세요. 수컷(`sex_Male`)의 계수는 무엇을 뜻하나요?

In [ ]:
# 문제 1 풀이

**문제 2.** 잘못 적힌 기록의 수를 1, 3, 6, 10건으로 늘려 보세요(거리는 `np.linspace(15, 19, k)`, 요금은 모두 2,000원). 건수마다 MSE(`LinearRegression`), MAE(`QuantileRegressor(quantile=0.5, alpha=0)`), 후버(`HuberRegressor()`)의 기울기와 절편을 구해 비교하세요.

In [ ]:
# 문제 2 풀이

**문제 3.** 택시 기록 50건 + 잘못 적힌 기록 3건에서 `HuberRegressor`의 `epsilon`을 1.35, 2.0, 3.0, 5.0으로 바꾸어 보세요. 기울기, 절편, `scale_`, 절댓값처럼 다룬 기록의 수(`outliers_.sum()`)가 어떻게 변하나요?

In [ ]:
# 문제 3 풀이

**문제 4.** 2절의 "세 특징 + 종" 모델(`reg2`)로 테스트 데이터의 잔차(실제 − 예측)를 구하고, 가로축에 예측값, 세로축에 잔차를 찍어 보세요. 잔차의 절댓값이 가장 큰 펭귄 3마리의 종, 성별, 몸무게를 출력하고, 모델이 왜 크게 틀렸을지 생각해 보세요.

In [ ]:
# 문제 4 풀이